In [1]:
!pip install pyspark

# Create DataFrame

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ParquetDemo").getOrCreate()

data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|
+--------+------------+---------+-----------+----------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|
+--------+------------+---------+-----------+----------------+



# Write Data as Parquet

In [3]:
df.write.mode("overwrite").parquet("/content/patient_parquet")

# Read Parquet Data

In [4]:
parquet_df = spark.read.parquet("/content/patient_parquet")
parquet_df.show()

+--------+------------+---------+-----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|
+--------+------------+---------+-----------+----------------+
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|
+--------+------------+---------+-----------+----------------+



# Schema Inspection

In [5]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



# Filtering Data

In [7]:
spark.read.parquet("/content/patient_parquet") \
    .filter("consultation_fee > 3000") \
    .show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
+--------+------------+---------+----------+----------------+



# Partitioned Parquet Write

In [8]:
df.write.mode("overwrite") \
    .partitionBy("city") \
    .parquet("/content/patient_parquet_partitioned")

# Read Partitioned Data

In [9]:
spark.read.parquet("/content/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



# Partition Pruning

In [10]:
spark.read.parquet("/content/patient_parquet_partitioned") \
    .filter("city = 'Hyderabad'") \
    .show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



# Append Mode

In [11]:
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]
new_df = spark.createDataFrame(new_data, columns)
new_df.write.mode("append").parquet("/content/patient_parquet")

# Overwrite Mode

In [12]:
df.write.mode("overwrite").parquet("/content/patient_parquet")

# Create SQL Table

In [14]:
df = spark.read.parquet("/content/patient_parquet")

df.createOrReplaceTempView("patient_parquet_table")

In [15]:
spark.sql("SELECT * FROM patient_parquet_table").show()

+--------+------------+---------+-----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|
+--------+------------+---------+-----------+----------------+
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|
+--------+------------+---------+-----------+----------------+



# Read Only One Partition

In [17]:
spark.read.parquet("/content/patient_parquet_partitioned") \
    .filter("city='Chennai'") \
    .show()

+--------+------------+----------+----------------+-------+
|visit_id|patient_name|department|consultation_fee|   city|
+--------+------------+----------+----------------+-------+
|     105|Vikram Singh| Neurology|            7000|Chennai|
+--------+------------+----------+----------------+-------+

